# Side Effect Unique263 Targets

Build final unique-263 targets while retaining the original split membership, row order and Morgan matrices. Requires the corrected linked dataset and existing Morgan feature directory.


## Stage 1

Source: `side_effect_and_baseline_history`, cell 15 (zero-based).


In [ ]:
# ============================================================
# BUILD FINAL 263-UNIQUE TARGETS
# Reuse SAME splits + SAME Morgan X matrices
# Only side-effect targets are updated
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import shutil
import json
import re
import unicodedata

# ============================================================
# PATHS
# ============================================================

FINAL_DATA = Path(
    "/content/drive/MyDrive/"
    "DDI_SIDE_EFFECT_84_CORRECTED/"
    "cleaned_ddi_sideeffects_84_263_UNIQUE.csv"
)

OLD_FEATURE_DIR = Path(
    "/content/drive/MyDrive/"
    "DDI_SIDE_EFFECT_84_CORRECTED/"
    "morgan_features_84"
)

NEW_FEATURE_DIR = Path(
    "/content/drive/MyDrive/"
    "DDI_SIDE_EFFECT_84_CORRECTED/"
    "morgan_features_84_263_UNIQUE"
)

NEW_FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOAD
# ============================================================

final_df = pd.read_csv(FINAL_DATA)

train_old = pd.read_csv(
    OLD_FEATURE_DIR / "train_rows.csv"
)

cal_old = pd.read_csv(
    OLD_FEATURE_DIR / "calibration_rows.csv"
)

test_old = pd.read_csv(
    OLD_FEATURE_DIR / "test_rows.csv"
)

print("=" * 100)
print("INPUT")
print("=" * 100)

print("Final dataset:", final_df.shape)

print(
    "Old split sizes:",
    len(train_old),
    len(cal_old),
    len(test_old)
)

assert len(train_old) == 134249
assert len(cal_old) == 28768
assert len(test_old) == 28768

# ============================================================
# 1) Verify each Interaction has exactly one new effect string
# ============================================================

interaction_effect_map = (
    final_df[
        ["Interaction", "sideeffect_str"]
    ]
    .drop_duplicates()
)

check = (
    interaction_effect_map
    .groupby("Interaction")["sideeffect_str"]
    .nunique()
)

assert check.max() == 1

interaction_to_effect = dict(
    zip(
        interaction_effect_map["Interaction"].astype(int),
        interaction_effect_map["sideeffect_str"].astype(str)
    )
)

print(
    "Interaction mappings:",
    len(interaction_to_effect)
)

assert len(interaction_to_effect) == 84


# ============================================================
# 2) Update sideeffect_str ONLY
#
# Keep exact split rows and exact row order.
# ============================================================

def update_split(frame):

    out = frame.copy()

    out["Interaction"] = (
        out["Interaction"]
        .astype(int)
    )

    out["sideeffect_str"] = (
        out["Interaction"]
        .map(interaction_to_effect)
    )

    if out["sideeffect_str"].isna().any():
        bad = out.loc[
            out["sideeffect_str"].isna(),
            "Interaction"
        ].unique()

        raise ValueError(
            f"Missing side-effect mapping for: {bad}"
        )

    return out


train_df = update_split(train_old)
cal_df = update_split(cal_old)
test_df = update_split(test_old)


print("\n Existing split membership preserved")
print(" Existing row order preserved")


# ============================================================
# 3) Build CLEAN 263-label vocabulary
# ============================================================

HIDDEN = re.compile(
    r"[\u200b\u200c\u200d\u2060\ufeff]"
)


def normalize_term(x):

    x = unicodedata.normalize(
        "NFKC",
        str(x)
    )

    x = HIDDEN.sub(
        "",
        x
    )

    x = re.sub(
        r"\s+",
        " ",
        x
    ).strip()

    return x


def parse_effects(value):

    output = []

    for x in str(value).split("|"):

        x = normalize_term(x)

        if x and x not in output:
            output.append(x)

    return output


VOCAB = sorted({

    term

    for value in final_df["sideeffect_str"]

    for term in parse_effects(value)

})


print("\n" + "=" * 100)
print("VOCABULARY")
print("=" * 100)

print(
    "Unique clean labels:",
    len(VOCAB)
)

assert len(VOCAB) == 263


normalized_vocab = [
    normalize_term(x).lower()
    for x in VOCAB
]

assert len(
    set(normalized_vocab)
) == 263


for required in [
    "tinnitus",
    "hyperglycemia",
    "anaphylaxis"
]:

    assert required in VOCAB

    print(
        required,
        ""
    )


# ============================================================
# 4) New label IDs
# ============================================================

label_to_id = {

    label: i

    for i, label
    in enumerate(VOCAB)

}


label_mapping = pd.DataFrame({

    "label_id":
        range(263),

    "label":
        VOCAB

})


# ============================================================
# 5) Build Y matrices
# ============================================================

def build_Y(frame):

    Y = np.zeros(
        (
            len(frame),
            263
        ),
        dtype=np.uint8
    )

    for row_id, value in enumerate(
        frame["sideeffect_str"]
    ):

        terms = parse_effects(
            value
        )

        for term in terms:

            Y[
                row_id,
                label_to_id[term]
            ] = 1

    return Y


print("\nBuilding final Y matrices...")

Y_train = build_Y(
    train_df
)

Y_cal = build_Y(
    cal_df
)

Y_test = build_Y(
    test_df
)


print(
    "Y_train:",
    Y_train.shape
)

print(
    "Y_cal:",
    Y_cal.shape
)

print(
    "Y_test:",
    Y_test.shape
)


assert Y_train.shape == (
    134249,
    263
)

assert Y_cal.shape == (
    28768,
    263
)

assert Y_test.shape == (
    28768,
    263
)


# ============================================================
# 6) Support QA
# ============================================================

train_support = (
    Y_train.sum(axis=0)
)

cal_support = (
    Y_cal.sum(axis=0)
)

test_support = (
    Y_test.sum(axis=0)
)


support_df = (
    label_mapping.copy()
)

support_df[
    "train_support"
] = train_support

support_df[
    "calibration_support"
] = cal_support

support_df[
    "test_support"
] = test_support


print("\n" + "=" * 100)
print("SUPPORT QA")
print("=" * 100)


print(
    "Minimum train support:",
    int(train_support.min())
)

print(
    "Minimum calibration support:",
    int(cal_support.min())
)

print(
    "Minimum test support:",
    int(test_support.min())
)


print("\nReplacement labels:")

print(
    support_df[
        support_df["label"].isin(
            [
                "tinnitus",
                "hyperglycemia",
                "anaphylaxis"
            ]
        )
    ].to_string(
        index=False
    )
)


# ============================================================
# 7) Save updated split CSVs
# ============================================================

train_df.to_csv(
    NEW_FEATURE_DIR /
    "train_rows.csv",
    index=False
)

cal_df.to_csv(
    NEW_FEATURE_DIR /
    "calibration_rows.csv",
    index=False
)

test_df.to_csv(
    NEW_FEATURE_DIR /
    "test_rows.csv",
    index=False
)


# ============================================================
# 8) Save NEW Y + mapping
# ============================================================

np.save(
    NEW_FEATURE_DIR /
    "Y_train.npy",
    Y_train
)

np.save(
    NEW_FEATURE_DIR /
    "Y_calibration.npy",
    Y_cal
)

np.save(
    NEW_FEATURE_DIR /
    "Y_test.npy",
    Y_test
)


label_mapping.to_csv(
    NEW_FEATURE_DIR /
    "label_mapping.csv",
    index=False
)


support_df.to_csv(
    NEW_FEATURE_DIR /
    "label_support_by_split.csv",
    index=False
)


# ============================================================
# 9) Reuse EXACT SAME X matrices
#
# Copy only — no Morgan regeneration.
# ============================================================

for filename in [
    "X_train.npz",
    "X_calibration.npz",
    "X_test.npz"
]:

    source = (
        OLD_FEATURE_DIR /
        filename
    )

    destination = (
        NEW_FEATURE_DIR /
        filename
    )

    if not destination.exists():

        print(
            "Copying:",
            filename
        )

        shutil.copy2(
            source,
            destination
        )

    else:

        print(
            "Already exists:",
            filename
        )


# ============================================================
# 10) Metadata
# ============================================================

metadata = {

    "dataset":
        str(FINAL_DATA),

    "interaction_classes":
        84,

    "side_effect_labels":
        263,

    "normalized_unique_labels":
        263,

    "replacement_labels": [
        "tinnitus",
        "hyperglycemia",
        "anaphylaxis"
    ],

    "hidden_character_labels":
        0,

    "empty_labels":
        0,

    "split_reused":
        True,

    "split_source":
        str(OLD_FEATURE_DIR),

    "morgan_features_reused":
        True,

    "feature_dimension":
        4096,

    "morgan_bits_per_drug":
        2048,

    "morgan_radius":
        2,

    "pair_order":
        "Drug1 || Drug2",

    "note":
        (
            "X matrices and split membership are "
            "identical to the prior corrected 84-class run. "
            "Only side-effect target vocabulary and Y matrices "
            "were rebuilt."
        )
}


with open(
    NEW_FEATURE_DIR /
    "feature_metadata.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# FINAL
# ============================================================

print("\n" + "=" * 110)
print("FINAL TARGET BUILD COMPLETE")
print("=" * 110)

print(
    "Feature directory:"
)

print(
    NEW_FEATURE_DIR
)

print(
    "\nTrain:",
    len(train_df),
    Y_train.shape
)

print(
    "Calibration:",
    len(cal_df),
    Y_cal.shape
)

print(
    "Test:",
    len(test_df),
    Y_test.shape
)

print(
    "\nLabels:",
    len(label_mapping)
)

print(
    "Normalized unique labels:",
    len(
        set(
            normalized_vocab
        )
    )
)

print("\n SAME X features")
print(" SAME train/cal/test rows")
print(" SAME Drug1/Drug2 order")
print(" SAME 84 Interaction classes")
print(" NEW 263 unique Side-Effect labels")
print(" NEW Y matrices")
print(" Ready for FINAL LightGBM retraining")